# Doctor Prescription AI — TrOCR Fine-tune (Kaggle)

End-to-end notebook: install → data → train → evaluate → push to HuggingFace.

**GPU:** enable T4 x2 or P100 in Kaggle settings before running.

## 1. Install

In [ ]:
!pip -q install transformers==4.42.3 datasets==2.20.0 evaluate==0.4.2 jiwer==3.0.4 \
  huggingface_hub==0.23.4 albumentations==1.4.10 sentencepiece==0.2.0

## 2. Imports & GPU check

In [ ]:
import os, torch, pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
from evaluate import load as load_metric
print('CUDA available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 3. Dataset — attach a Kaggle handwriting dataset then build manifest

In [ ]:
DATA_DIR = Path('/kaggle/input/your-prescription-dataset')  # <-- change
IMG_DIR  = DATA_DIR / 'images'
MANIFEST = DATA_DIR / 'labels.csv'  # must have columns: image,text

df = pd.read_csv(MANIFEST)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
split = int(len(df) * 0.9)
train_df, val_df = df.iloc[:split], df.iloc[split:]
print(len(train_df), len(val_df))

In [ ]:
class OCRDataset(Dataset):
    def __init__(self, df, processor, img_dir, max_len=128):
        self.df = df.reset_index(drop=True); self.processor = processor
        self.img_dir = img_dir; self.max_len = max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_dir / row['image']).convert('RGB')
        pix = self.processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        lab = self.processor.tokenizer(row['text'], padding='max_length', max_length=self.max_len,
                                       truncation=True, return_tensors='pt').input_ids.squeeze(0)
        lab[lab == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pix, 'labels': lab}

## 4. Model

In [ ]:
BASE = 'microsoft/trocr-base-handwritten'
processor = TrOCRProcessor.from_pretrained(BASE)
model = VisionEncoderDecoderModel.from_pretrained(BASE)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 128; model.config.num_beams = 4

## 5. Train

In [ ]:
train_ds = OCRDataset(train_df, processor, IMG_DIR)
val_ds   = OCRDataset(val_df,   processor, IMG_DIR)
cer = load_metric('cer'); wer = load_metric('wer')
def metrics(pred):
    ids = pred.predictions; lab = pred.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    ps = processor.batch_decode(ids, skip_special_tokens=True)
    ls = processor.batch_decode(lab, skip_special_tokens=True)
    return {'cer': cer.compute(predictions=ps, references=ls), 'wer': wer.compute(predictions=ps, references=ls)}

args = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/out',
    num_train_epochs=8, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, learning_rate=5e-5, weight_decay=0.01, warmup_ratio=0.1,
    fp16=torch.cuda.is_available(), eval_strategy='steps', eval_steps=500,
    save_strategy='steps', save_steps=500, logging_steps=50, save_total_limit=2,
    predict_with_generate=True, generation_max_length=128,
    metric_for_best_model='cer', greater_is_better=False, load_best_model_at_end=True,
    report_to=['none'], seed=42,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                         data_collator=default_data_collator, tokenizer=processor.feature_extractor,
                         compute_metrics=metrics)
trainer.train()

## 6. Save best + push to HuggingFace

In [ ]:
BEST = '/kaggle/working/out/best'
trainer.save_model(BEST); processor.save_pretrained(BEST)

# HuggingFace push — add HF_TOKEN as a Kaggle Secret first
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
REPO = 'your-org/trocr-prescription-v1'
model.push_to_hub(REPO); processor.push_to_hub(REPO)
print('Pushed', REPO)